In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'Scripts'))  # helper modules (run_config, obs, standardplots) live in LPNE/Scripts

from run_config import PROJECT  # project root (LPNE/), found relative to run_config.py - no hard-coded paths
import run_config as rc
import standardplots as sp
import emusetup_and_mc as emc   # shared emulator build and Monte Carlo rollout helpers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import detrend

### Long rollout over the whole LR04 record
The 30 Monte Carlo models of AblationTest setup 3c (state: X = dO18 (LR04) only; forcing: eccentricity, precession,
obliquity), rolled out from the oldest window of the LR04 record (5320 ka) to the present. The models were trained on
1170-234 ka (mid start), so everything older than 1461 ka is far outside anything they have seen.

**Forcing:** the models were trained on the Berger (1990) orbital elements, but that file stops at 1500 ka. The
rollout is forced with the Laskar (2004) elements over the whole record instead, so there is no jump in the forcing
inside the record. Over 0-1500 ka the two agree closely for eccentricity and precession (r = 0.996, 0.98) and less
closely for obliquity (r = 0.96).

In [ ]:
# ---- setup 3c's models (the same config / seeds / folder as in AblationTest) ----
DATASET = rc.DATASET        # '1461ka'
TRAIN_START = 'mid'
PARAMS = {'conv filters': 30, 'conv kernel': 10, 'dense unit': 28,
          'dropout rate': 0.3353763108415839, 'learning rate': 0.00038902232764492545,
          'dwt gamma': 0.6506584806454648, 'dwt beta': 0.4637125670415251, 'dwt eps': 4.1664542731895874e-06,
          'dwt levels': 2}  # AblationTest's PARAMS (the 'mid' start optimization set)

CFG3c_KW = dict(train_start=TRAIN_START, state=('X',), forcing=('ecc', 'prec', 'obl'))
cfg3c = rc.make_config(DATASET, seed=42, params=PARAMS, **CFG3c_KW)
MC_SEEDS = list(range(30))
MC_DIR = rc.RUNS_DIR / '_Ablations' / 'setup3c_MC'
MC_PATHS = emc.mc_paths(DATASET, CFG3c_KW, MC_SEEDS, MC_DIR, PARAMS)
missing = [s for s, p in MC_PATHS.items() if not p['weights'].exists()]
assert not missing, f"no weights for seeds {missing} in {MC_DIR}: train them in AblationTest (setup 3c) first"

# the scaler every setup 3c model was trained with (fit on the mid start's training split, Berger elements)
_, scaler3c, _ = rc.load_run(MC_PATHS[0]['dir'].name, MC_DIR)
W = cfg3c['window_size']
print("inputs:", rc.inputs_label(cfg3c), "| models:", len(MC_PATHS), "| window:", W)

In [ ]:
# ---- the whole LR04 record (GP mean) with the Laskar orbital elements, oldest -> youngest ----
cfg_la = rc.make_config(DATASET, seed=42, params=PARAMS, train_start=TRAIN_START, state=('X',),
                        forcing=('ecc_la04', 'prec_la04', 'obl_la04'))   # only used for its forcing names
gpr = pd.read_csv(rc.GPR_CSV, index_col=0).sort_values(rc.AGE_COL, ascending=False).reset_index(drop=True)
ages = gpr[rc.AGE_COL].to_numpy(dtype=float)              # 5320 -> 0 ka
record = np.column_stack([gpr['dO18_LR04'].to_numpy(), rc.forcing_record(cfg_la, ages)])   # X, ecc, prec, obl
assert not np.isnan(record).any()
record_scaled = scaler3c.transform(record)
print(f"record: {ages[0]:.0f} -> {ages[-1]:.0f} ka, {len(ages)} steps")

In [ ]:
# ---- roll every model out from the record's oldest window (cached: ~30 x 5300 steps) ----
# rollout_mc takes a stack of initial-condition records; here there's one, the LR04 record itself
CACHE = rc.RUNS_DIR / '_LongSpectra' / 'setup3c_la04_rollouts.npz'
RERUN = False
if CACHE.exists() and not RERUN:
    roll_phys = np.load(CACHE)['roll_phys']
else:
    _, roll = emc.rollout_mc(cfg3c, PARAMS, MC_PATHS, record_scaled[None])     # (n_models, 1, n_ages, 4), scaled
    roll_phys = np.stack([scaler3c.inverse_transform(r[0]) for r in roll])     # (n_models, n_ages, 4), physical
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    np.savez(CACHE, roll_phys=roll_phys, ages=ages, seeds=MC_SEEDS)
X_emu = roll_phys[:, :, 0]          # (n_models, n_ages): dO18 [permil]
X_obs = record[:, 0]
print("rollouts:", roll_phys.shape)

In [ ]:
# ---- the rollouts against LR04, with the two spectral windows shaded ----
PERIODS = {'3.0-1.5 Ma': (3000, 1500), '1.0-0 Ma': (1000, 0)}   # label -> (oldest, youngest) age [ka]
PERIOD_COLORS = {'3.0-1.5 Ma': '#D6195E', '1.0-0 Ma': '#1E88E5'}
TRAIN_AGES = (cfg3c['training_start'], 0)                       # training + validation ages [ka]

with plt.rc_context({'font.size': 16}):
    fig, ax = plt.subplots(figsize=(22, 7))
    for lab, (old, young) in PERIODS.items():
        ax.axvspan(old, young, color=PERIOD_COLORS[lab], alpha=0.08, lw=0)
        ax.annotate(lab, xy=((old + young) / 2, 1.0), xycoords=('data', 'axes fraction'), xytext=(0, -6),
                    textcoords='offset points', ha='center', va='top', fontsize=14, color=PERIOD_COLORS[lab])
    ax.plot(ages, X_emu.T, color='#FFB600', lw=0.6, alpha=0.15)
    ax.plot([], [], color='#FFB600', lw=2, alpha=0.5, label=f"emulator rollouts ({len(X_emu)} models)")
    ax.plot(ages, X_emu.mean(axis=0), color='#B07D00', lw=1.8, label="ensemble mean")
    ax.plot(ages, X_obs, color='black', lw=1.2, alpha=0.8, label="LR04 (GP mean)")
    ax.axvline(TRAIN_AGES[0], color='dimgray', linestyle=':', lw=2)
    ax.annotate('training / validation  →', xy=(TRAIN_AGES[0], 0.02), xycoords=('data', 'axes fraction'),
                xytext=(-6, 0), textcoords='offset points', ha='left', va='bottom', fontsize=13, color='dimgray')
    ax.invert_xaxis()   # also flips the arrow direction: it points toward the present
    ax.invert_yaxis()   # dO18 convention: glacial (high dO18) down
    ax.set_xlabel("age [ka BP]")
    ax.set_ylabel(r"$\delta^{18}$O [‰]")
    ax.set_title("Setup 3c (X + ecc/prec/obl) rolled out over the whole LR04 record, Laskar forcing", fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower left', fontsize=14)
    plt.tight_layout()
    plt.show()

### Spectra: 3.0-1.5 Ma vs. 1.0-0 Ma
Multitaper spectra of dO18 in each window, in physical units (permil^2 kyr), so the two windows are on the same
scale. Each window is linearly detrended first: the long-term increase in dO18 over 3.0-1.5 Ma would otherwise leak
into the lowest frequencies. The spectrum is a density, so windows of different length (1500 vs. 1000 kyr) are
comparable; the shorter window has coarser frequency resolution.

Emulator: the spectrum of each of the 30 models' rollouts, summarized by the median and the 5-95% range over the
models (not the spectrum of the ensemble mean, which averages out variability that isn't in phase across models).

In [ ]:
# ---- spectra of each window: LR04 (with its 95% chi^2 CI) and the emulator ensemble ----
NW = 2
def window(x, old, young):
    """the part of x (last axis along `ages`) between ages old and young [ka], inclusive, linearly detrended"""
    sel = (ages <= old) & (ages >= young)
    return detrend(x[..., sel], axis=-1)

spec = {}   # period -> {'freq', 'obs', 'obs_lo', 'obs_hi', 'emu' (n_models, n_freq)}
for lab, (old, young) in PERIODS.items():
    f, s, lo, hi, _ = sp.multitaper_spectrum(window(X_obs, old, young), cfg3c['timestep'], NW)
    emu = np.array([sp.multitaper_spectrum(xm, cfg3c['timestep'], NW)[1] for xm in window(X_emu, old, young)])
    spec[lab] = {'freq': f, 'obs': s, 'obs_lo': lo, 'obs_hi': hi, 'emu': emu}

with plt.rc_context({'font.size': 16}):
    fig, axs = plt.subplots(1, 2, figsize=(22, 8), sharex=True, sharey=True)
    for lab, d in spec.items():
        c, pos = PERIOD_COLORS[lab], d['freq'] > 0
        f = d['freq'][pos]
        axs[0].plot(f, d['obs'][pos], color=c, lw=2, label=lab)
        axs[0].fill_between(f, d['obs_lo'][pos], d['obs_hi'][pos], color=c, alpha=0.15, lw=0)
        q05, q50, q95 = np.percentile(d['emu'][:, pos], [5, 50, 95], axis=0)
        axs[1].plot(f, q50, color=c, lw=2, label=lab)
        axs[1].fill_between(f, q05, q95, color=c, alpha=0.15, lw=0)

    for ax, title, band in zip(axs, ["LR04 (GP mean)", f"Emulator, setup 3c ({len(X_emu)} models)"],
                               [r"bands: 95% $\chi^2$ CI", "lines: median over models   bands: 5-95% over models"]):
        for period, color in sp.ORBITAL.items():
            ax.axvline(1 / period, color=color, linestyle='dashed', alpha=0.6, label=f'{period} kyr')
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlabel('frequency [1/kyr]')
        ax.secondary_xaxis('top', functions=(sp._inv, sp._inv)).set_xlabel('period [kyr]')
        ax.set_title(title, fontweight='bold')
        ax.text(0.99, 0.01, band, transform=ax.transAxes, ha='right', va='bottom', fontsize=13, color='dimgray')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='lower left', fontsize=14)
    axs[0].set_ylabel(r'power [‰$^2$ kyr]')
    fig.suptitle(r"Multitaper spectra of $\delta^{18}$O: 3.0-1.5 Ma vs. 1.0-0 Ma", fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# ---- share of each window's variance in the orbital bands (detrended; f = 0 excluded) ----
# emulator: median [5%, 95%] over the models
BANDS = {'100 kyr (80-125)': (80, 125), '41 kyr (37-46)': (37, 46), '23 kyr (19-24)': (19, 24)}
def band_share(f, s):
    """fraction of the total power (f > 0) in each band"""
    pos = f > 0
    total = s[..., pos].sum(axis=-1)
    return {b: s[..., (f >= 1 / hi) & (f <= 1 / lo)].sum(axis=-1) / total for b, (lo, hi) in BANDS.items()}

rows = {}
for lab, d in spec.items():
    obs_share, emu_share = band_share(d['freq'], d['obs']), band_share(d['freq'], d['emu'])
    for b in BANDS:
        q05, q50, q95 = np.percentile(emu_share[b], [5, 50, 95])
        rows[(lab, b)] = {'LR04': f"{obs_share[b]:.2f}", 'emulator': f"{q50:.2f} [{q05:.2f}, {q95:.2f}]"}
pd.DataFrame(rows).T.rename_axis(['window', 'band'])